# 05 - Implicit Alternating Least Squares (ALS) Collaborative Filtering
## E-Commerce Recommendation System

**Objective:** Train and evaluate matrix factorization using Implicit ALS on sparse user-item interaction matrix.

In [1]:
import sys, pickle
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.collaborative_filtering import ALSRecommender
from src.features.build_features import build_sparse_interaction_matrix

train_user_item = pd.read_parquet(ROOT / 'data/processed/train.parquet')
print(f'Train events: {len(train_user_item):,}')


Train events: 1,928,948


In [2]:
# Build interaction matrix and train ALS
from src.features.build_features import aggregate_user_item_interactions
agg_train = aggregate_user_item_interactions(train_user_item)
mat, mappings = build_sparse_interaction_matrix(agg_train)

als = ALSRecommender(factors=64, regularization=0.1, iterations=20, random_state=42)
als.fit(mat, mappings)


2026-11-20 02:56:41,204 [INFO] Aggregating 1928948 event interactions by user-product pairs...
2026-11-20 02:56:43,574 [INFO] Generated 1496538 unique user-product interaction records
2026-11-20 02:56:43,591 [INFO] Building CSR interaction matrix for 1496538 interactions...
2026-11-20 02:56:44,735 [INFO] Sparse Matrix built: 978906 users x 200974 items | Non-zeros: 1496538 | Sparsity: 99.9992%
2026-11-20 02:56:44,737 [INFO] Initializing ALS model (factors=64, reg=0.100, iter=20)...
d:\projects & datasets\Ecommerce_rcmd_sys\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
2026-11-20 02:56:44,781 [INFO] Fitting ALS on user-item matrix of shape (978906, 2009

  0%|          | 0/20 [00:00<?, ?it/s]

2026-11-20 02:57:31,439 [INFO] ALS model training completed successfully.


In [3]:
# Test recommendation for warm user
sample_visitor = list(mappings['visitorid_to_idx'].keys())[10]
recs = als.recommend(sample_visitor, top_n=10)
print(f'Personalized ALS recommendations for Visitor {sample_visitor}:')
for r, (iid, score) in enumerate(recs, 1):
    print(f'  Rank {r:2d}: Item {iid} (Predicted score: {score:.4f})')


Personalized ALS recommendations for Visitor 19:
  Rank  1: Item 339703 (Predicted score: 0.0000)
  Rank  2: Item 316753 (Predicted score: 0.0000)
  Rank  3: Item 65273 (Predicted score: 0.0000)
  Rank  4: Item 315545 (Predicted score: 0.0000)
  Rank  5: Item 344723 (Predicted score: 0.0000)
  Rank  6: Item 253615 (Predicted score: 0.0000)
  Rank  7: Item 320494 (Predicted score: 0.0000)
  Rank  8: Item 422425 (Predicted score: 0.0000)
  Rank  9: Item 22436 (Predicted score: 0.0000)
  Rank 10: Item 198209 (Predicted score: 0.0000)
